# Sel 1: Impor dan konfigurasi

In [1]:
import json
import shutil
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd

CARD_VERSION = "4.0.2"
CARD_URL = f"https://card.mcmaster.ca/download/0/broadstreet-v{CARD_VERSION}.tar.bz2"

CARD_DIR = Path("data_card")
CARD_DIR.mkdir(exist_ok=True)
CARD_TAR = CARD_DIR / f"card-data-v{CARD_VERSION}.tar.bz2"
CARD_EXTRACT = CARD_DIR / f"card-v{CARD_VERSION}"

OUT_DIR = Path("results_card")
OUT_DIR.mkdir(exist_ok=True)

# Sel 2: Unduh dan ekstrak

In [2]:
if not CARD_TAR.exists():
    req = urllib.request.Request(CARD_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=180) as r, open(CARD_TAR, "wb") as f:
        shutil.copyfileobj(r, f)
print("Ukuran arsip (MB):", round(CARD_TAR.stat().st_size / 1e6, 1))

if not CARD_EXTRACT.exists():
    CARD_EXTRACT.mkdir()
    with tarfile.open(CARD_TAR, "r:bz2") as t:
        try:
            t.extractall(CARD_EXTRACT, filter="data")
        except TypeError:
            t.extractall(CARD_EXTRACT)

for p in sorted(CARD_EXTRACT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(CARD_EXTRACT)}")

CARD_JSON = next(CARD_EXTRACT.rglob("card.json"))
print("card.json:", CARD_JSON)

Ukuran arsip (MB): 4.4
    0.00 MB  CARD-Download-README.txt
    0.22 MB  PMID.tsv
    0.04 MB  aro_categories.tsv
    0.74 MB  aro_categories_index.tsv
    1.13 MB  aro_index.tsv
   38.09 MB  card.json
    6.29 MB  nucleotide_fasta_protein_homolog_model.fasta
    0.02 MB  nucleotide_fasta_protein_knockout_model.fasta
    0.01 MB  nucleotide_fasta_protein_overexpression_model.fasta
    0.43 MB  nucleotide_fasta_protein_variant_model.fasta
    0.19 MB  nucleotide_fasta_rRNA_gene_variant_model.fasta
    2.33 MB  protein_fasta_protein_homolog_model.fasta
    0.01 MB  protein_fasta_protein_knockout_model.fasta
    0.00 MB  protein_fasta_protein_overexpression_model.fasta
    0.15 MB  protein_fasta_protein_variant_model.fasta
    0.00 MB  shortname_antibiotics.tsv
    0.00 MB  shortname_pathogens.tsv
    0.65 MB  snps.txt
card.json: data_card/card-v4.0.2/card.json


# Sel 3: Muat dan periksa tipe model

In [3]:
with open(CARD_JSON, encoding="utf-8") as f:
    card = json.load(f)

models = {k: v for k, v in card.items() if isinstance(v, dict) and "ARO_category" in v}
print("Jumlah model:", len(models))
print(Counter(v.get("model_type", "NA") for v in models.values()))

Jumlah model: 6451
Counter({'protein homolog model': 6059, 'protein variant model': 239, 'rRNA gene variant model': 86, 'efflux pump system meta-model': 22, 'protein knockout model': 16, 'protein overexpression model': 15, 'gene cluster meta-model': 14})


# Sel 4: Susun tabel sekuens

In [4]:
def drug_classes(entry):
    return sorted({
        c.get("category_aro_name", "").strip()
        for c in entry.get("ARO_category", {}).values()
        if c.get("category_aro_class_name") == "Drug Class"
        and c.get("category_aro_name", "").strip()
    })

rows = []
for model_id, e in models.items():
    dcs = drug_classes(e)
    for seq_id, info in e.get("model_sequences", {}).get("sequence", {}).items():
        s = info.get("protein_sequence", {}).get("sequence", "")
        if s:
            rows.append({
                "model_id": model_id,
                "seq_id": seq_id,
                "aro_name": str(e.get("ARO_name")),
                "model_type": e.get("model_type"),
                "drug_classes": dcs,
                "sequence": s.strip().upper().rstrip("*"),
            })

df = pd.DataFrame(rows)
print(df.shape)
print(df["model_type"].value_counts(dropna=False))

(6329, 6)
model_type
protein homolog model           6059
protein variant model            239
protein knockout model            16
protein overexpression model      15
Name: count, dtype: int64


# Sel 5: Saring, bersihkan, dan deduplikasi

In [5]:
# Cocokkan dengan nama persis dari output Sel 3
MODEL_TYPES_KEEP = ["protein homolog model"]

std = set("ACDEFGHIKLMNPQRSTVWY")
d = df[df["model_type"].isin(MODEL_TYPES_KEEP) & (df["drug_classes"].str.len() > 0)].copy()
d = d[d["sequence"].apply(lambda s: len(set(s) - std) == 0)]
print("Setelah penyaringan:", len(d), "| sekuens unik:", d["sequence"].nunique())

d["dc_key"] = d["drug_classes"].apply(";".join)
conf = d.groupby("sequence")["dc_key"].nunique()
print("Sekuens identik dengan set kelas obat berbeda:", int((conf > 1).sum()))

agg = {}
for r in d.itertuples():
    a = agg.setdefault(r.sequence, {"classes": set(), "names": set(), "n": 0})
    a["classes"].update(r.drug_classes)
    a["names"].add(r.aro_name)
    a["n"] += 1

dedup = pd.DataFrame([
    {"sequence": s, "drug_classes": sorted(a["classes"]),
     "aro_names": sorted(a["names"]), "n_entries": a["n"]}
    for s, a in agg.items()
])
print(dedup.shape)

Setelah penyaringan: 6054 | sekuens unik: 6054
Sekuens identik dengan set kelas obat berbeda: 0
(6054, 4)


# Sel 6: Matriks label dan pembuangan kelas langka

In [6]:
MIN_PER_CLASS = 10

cnt = Counter(c for l in dedup["drug_classes"] for c in l)
keep = sorted(c for c, n in cnt.items() if n >= MIN_PER_CLASS)
print(len(cnt), "kelas total ->", len(keep), f"kelas dengan minimal {MIN_PER_CLASS} sekuens")

idx = {c: i for i, c in enumerate(keep)}
dedup["labels"] = dedup["drug_classes"].apply(lambda l: [idx[c] for c in l if c in idx])
dedup = dedup[dedup["labels"].str.len() > 0].reset_index(drop=True)
print("Sekuens tersisa:", len(dedup))
print("Rata-rata label per sekuens:", round(dedup["labels"].str.len().mean(), 2))
print("Panjang sekuens:")
print(dedup["sequence"].str.len().describe())

39 kelas total -> 27 kelas dengan minimal 10 sekuens
Sekuens tersisa: 6042
Rata-rata label per sekuens: 2.13
Panjang sekuens:
count    6042.000000
mean      319.672459
std        99.626811
min        53.000000
25%       273.000000
50%       290.000000
75%       381.000000
max      1452.000000
Name: sequence, dtype: float64


# Sel 7: Simpan (FASTA polos, label terpisah, pemetaan bersama versi)

In [7]:
dedup["seq_uid"] = [f"card{CARD_VERSION}_{i:06d}" for i in range(len(dedup))]

with open(OUT_DIR / "card_proteins.fasta", "w") as f:
    for r in dedup.itertuples():
        f.write(f">{r.seq_uid}\n{r.sequence}\n")

out = dedup.assign(
    drug_classes=dedup["drug_classes"].apply(";".join),
    aro_names=dedup["aro_names"].apply(";".join),
    labels=dedup["labels"].apply(lambda l: ";".join(map(str, l))),
)
out.to_csv(OUT_DIR / "card_proteins_labels.csv", index=False)

with open(OUT_DIR / "drug_class_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"card_version": CARD_VERSION, "classes": idx}, f, indent=2, ensure_ascii=False)

print("Tersimpan di:", OUT_DIR.resolve())

Tersimpan di: /Users/lilis/Documents/Work/BRIN/tugas-akhir/data/results_card


# Sel 8: Pratinjau tabel

In [8]:
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
dedup.head(10)

,sequence,drug_classes,aro_names,n_entries,labels,seq_uid
0,MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRY...,[cephalosporin],[CblA-1],1,[4],card4.0.2_000000
1,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAW...,"[cephalosporin, penicillin beta-lactam]",[SHV-52],1,"[4, 18]",card4.0.2_000001
2,MIGLIVARSKNNVIGKNGNIPWKIKGEQKQFRELTTGNVVIMGRKSYEEIGHPLPN...,[diaminopyrimidine antibiotic],[dfrF],1,[5],card4.0.2_000002
3,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGGRLGVALIDTA...,"[azole antibiotic, cephalosporin, imidazole antibiotic]",[CTX-M-130],1,"[2, 4, 10]",card4.0.2_000003
4,MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRFGDLVFRQLAP...,"[carbapenem, cephalosporin, penicillin beta-lactam]",[NDM-6],1,"[3, 4, 18]",card4.0.2_000004
5,MMKKSLCCALLLGISCSALATPVSEKQLAEVVANTVTPLMKAQSVPGMAVAVIYQG...,"[carbapenem, cephalosporin, penicillin beta-lactam]",[ACT-35],1,"[3, 4, 18]",card4.0.2_000005
6,MNVRKHKASFFSVVITFLCLTLSLNANATDSVLEAVTNAETELGARIGLAVHDLET...,[penicillin beta-lactam],[CARB-5],1,[18],card4.0.2_000006
7,MTKKMNKYNGKKLSRGEPPNFSGQHFMHNKRLLKEIVDKADVSVRDTVLELGAGKG...,"[lincosamide antibiotic, macrolide antibiotic, streptogr...",[Erm(34)],1,"[11, 12, 24]",card4.0.2_000007
8,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILE...,"[cephalosporin, monobactam, penicillin beta-lactam]",[TEM-126],1,"[4, 13, 18]",card4.0.2_000008
9,MNVQNCMVKAVSVSIILFASLSLAAQKVKEPTVSNADWSKPYRPFRIAGNLYYIGT...,"[cephalosporin, penicillin beta-lactam]",[LRA-12],1,"[4, 18]",card4.0.2_000009


# Sel 9: Satu contoh baris lengkap

In [9]:
inv = {i: c for c, i in idx.items()}
r = dedup.iloc[0]
print("seq_uid      :", r["seq_uid"] if "seq_uid" in dedup.columns else "(jalankan Sel 7 dulu)")
print("aro_names    :", r["aro_names"])
print("drug_classes :", r["drug_classes"])
print("labels (idx) :", r["labels"], "->", [inv[i] for i in r["labels"]])
print("n_entries    :", r["n_entries"])
print("panjang      :", len(r["sequence"]), "aa")
print("sequence     :", r["sequence"][:80], "...")

seq_uid      : card4.0.2_000000
aro_names    : ['CblA-1']
drug_classes : ['cephalosporin']
labels (idx) : [4] -> ['cephalosporin']
n_entries    : 1
panjang      : 296 aa
sequence     : MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKM ...


# Sel 10: Distribusi sekuens per kelas obat

In [10]:
cls_count = Counter(i for l in dedup["labels"] for i in l)
cls_df = (pd.DataFrame({"kelas_obat": [inv[i] for i in cls_count],
                        "jumlah_sekuens": list(cls_count.values())})
          .sort_values("jumlah_sekuens", ascending=False)
          .reset_index(drop=True))
cls_df

,kelas_obat,jumlah_sekuens
0,cephalosporin,3390
1,penicillin beta-lactam,3102
2,carbapenem,2561
3,monobactam,1300
4,azole antibiotic,267
5,imidazole antibiotic,267
6,aminoglycoside antibiotic,239
7,peptide antibiotic,234
8,nonribosomal peptide antibiotics,227
9,fluoroquinolone antibiotic,225


# Sel 11: Distribusi jumlah label per sekuens

In [11]:
dedup["labels"].str.len().value_counts().sort_index()

labels
1     2094
2     1606
3     2006
4      279
5       21
6        3
7        2
8       12
9        5
10       8
13       3
14       2
15       1
Name: count, dtype: int64